# 23. π0 and FAST — VLA flow policy and action tokenizer

π0 keeps the released training/inference structure at reduced width and token
counts: SigLIP-style image tokens, language prefix, state/action suffix,
block-causal mask, Beta-time flow matching and Euler rollout.

FAST uses a tokenizer fitted **once on a training corpus**. DCT coefficients are
flattened in frequency-major order `[frequency, action_dimension]`; quantization
symbols are then compressed with corpus-trained BPE merge rules which are frozen
and reused for unseen action chunks. No per-sample BPE fitting is performed.


In [ ]:
import math
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F


torch.manual_seed(12)
device = torch.device("cpu")


## 1. π0 attention mask and flow target


In [ ]:
def block_attention_mask(valid, ar_mask):
    cumulative = torch.cumsum(ar_mask.long(), dim=1)
    causal = cumulative[:, None, :] <= cumulative[:, :, None]
    valid_2d = valid[:, None, :] & valid[:, :, None]
    return causal & valid_2d


def pi0_time_embedding(
    t,
    dim,
    min_period=4e-3,
    max_period=4.0,
):
    half = dim // 2
    fraction = torch.linspace(
        0,
        1,
        half,
        device=t.device,
    )
    period = min_period * (
        max_period / min_period
    ) ** fraction
    angle = t[:, None] * (2 * math.pi / period)[None]
    return torch.cat([angle.sin(), angle.cos()], dim=-1)


def apply_rope(x, position):
    dim = x.size(-1)
    pair_index = torch.arange(
        0,
        dim,
        2,
        device=x.device,
        dtype=torch.float32,
    )
    inverse_frequency = 1.0 / (
        10000 ** (pair_index / dim)
    )
    angle = (
        position[:, None, :, None].float()
        * inverse_frequency[None, None, None]
    )
    even = x[..., 0::2]
    odd = x[..., 1::2]
    rotated_even = even * angle.cos() - odd * angle.sin()
    rotated_odd = even * angle.sin() + odd * angle.cos()
    return torch.stack(
        [rotated_even, rotated_odd],
        dim=-1,
    ).flatten(-2)


## 2. Small-width π0 joint prefix/action-expert transformer


In [ ]:
class VisionBlock(nn.Module):
    def __init__(self, dim=32, heads=16):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(
            dim,
            heads,
            batch_first=True,
        )
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(),
            nn.Linear(4 * dim, dim),
        )

    def forward(self, x):
        normalized = self.norm1(x)
        attended, _ = self.attn(
            normalized,
            normalized,
            normalized,
            need_weights=False,
        )
        x = x + attended
        return x + self.mlp(self.norm2(x))


class SigLIPVision(nn.Module):
    def __init__(
        self,
        dim=32,
        image_size=28,
        patch_size=14,
        depth=27,
    ):
        super().__init__()
        self.patch = nn.Conv2d(
            3,
            dim,
            patch_size,
            stride=patch_size,
        )
        patch_count = (image_size // patch_size) ** 2
        self.position = nn.Parameter(
            torch.randn(1, patch_count, dim) * 0.02
        )
        self.blocks = nn.ModuleList(
            [VisionBlock(dim, 16) for _ in range(depth)]
        )
        self.norm = nn.LayerNorm(dim)

    def forward(self, image):
        hidden = self.patch(image).flatten(2).transpose(1, 2)
        hidden = hidden + self.position[:, : hidden.size(1)]
        for block in self.blocks:
            hidden = block(hidden)
        return self.norm(hidden)


class GemmaFFN(nn.Module):
    def __init__(self, dim=32, hidden_dim=128):
        super().__init__()
        self.gate = nn.Linear(dim, hidden_dim, bias=False)
        self.value = nn.Linear(dim, hidden_dim, bias=False)
        self.output = nn.Linear(hidden_dim, dim, bias=False)

    def forward(self, x):
        gate = F.gelu(
            self.gate(x),
            approximate="tanh",
        )
        value = self.value(x)
        return self.output(gate * value)


class Pi0Layer(nn.Module):
    def __init__(self, dim=32, q_heads=8, kv_heads=1):
        super().__init__()
        assert q_heads % kv_heads == 0
        self.q_heads = q_heads
        self.kv_heads = kv_heads
        self.head_dim = dim // q_heads
        self.repeat = q_heads // kv_heads

        self.prefix_attn_norm = nn.RMSNorm(dim)
        self.suffix_attn_norm = nn.RMSNorm(dim)
        self.prefix_q = nn.Linear(
            dim,
            q_heads * self.head_dim,
            bias=False,
        )
        self.prefix_kv = nn.Linear(
            dim,
            2 * kv_heads * self.head_dim,
            bias=False,
        )
        self.suffix_q = nn.Linear(
            dim,
            q_heads * self.head_dim,
            bias=False,
        )
        self.suffix_kv = nn.Linear(
            dim,
            2 * kv_heads * self.head_dim,
            bias=False,
        )
        self.prefix_out = nn.Linear(dim, dim, bias=False)
        self.suffix_out = nn.Linear(dim, dim, bias=False)
        self.prefix_ffn_norm = nn.RMSNorm(dim)
        self.suffix_ffn_norm = nn.RMSNorm(dim)
        self.prefix_ffn = GemmaFFN(dim, 4 * dim)
        self.suffix_ffn = GemmaFFN(dim, 4 * dim)

    def project(self, x, q_projection, kv_projection, position):
        batch, length, _ = x.shape
        query = q_projection(x).view(
            batch,
            length,
            self.q_heads,
            self.head_dim,
        ).transpose(1, 2)
        kv = kv_projection(x).view(
            batch,
            length,
            2,
            self.kv_heads,
            self.head_dim,
        )
        key, value = kv.permute(2, 0, 3, 1, 4).unbind(0)
        query = apply_rope(query, position)
        key = apply_rope(key, position)
        key = key.repeat_interleave(self.repeat, dim=1)
        value = value.repeat_interleave(self.repeat, dim=1)
        return query, key, value

    def forward(self, prefix, suffix, mask, position):
        split = prefix.size(1)
        prefix_n = self.prefix_attn_norm(prefix)
        suffix_n = self.suffix_attn_norm(suffix)

        pq, pk, pv = self.project(
            prefix_n,
            self.prefix_q,
            self.prefix_kv,
            position[:, :split],
        )
        sq, sk, sv = self.project(
            suffix_n,
            self.suffix_q,
            self.suffix_kv,
            position[:, split:],
        )
        query = torch.cat([pq, sq], dim=2)
        key = torch.cat([pk, sk], dim=2)
        value = torch.cat([pv, sv], dim=2)
        query = query * (self.head_dim ** -0.5)

        additive_mask = torch.zeros_like(
            mask,
            dtype=query.dtype,
        )
        additive_mask = additive_mask.masked_fill(
            ~mask,
            torch.finfo(query.dtype).min,
        )
        attended = F.scaled_dot_product_attention(
            query,
            key,
            value,
            attn_mask=additive_mask[:, None],
            scale=1.0,
        )
        attended = attended.transpose(1, 2).contiguous().flatten(2)

        prefix = prefix + self.prefix_out(attended[:, :split])
        suffix = suffix + self.suffix_out(attended[:, split:])
        prefix = prefix + self.prefix_ffn(
            self.prefix_ffn_norm(prefix)
        )
        suffix = suffix + self.suffix_ffn(
            self.suffix_ffn_norm(suffix)
        )
        return prefix, suffix


class Pi0(nn.Module):
    def __init__(self, dim=32, action_dim=3, horizon=4, vocab=64):
        super().__init__()
        self.horizon = horizon
        self.action_dim = action_dim
        self.dim = dim
        self.vision = SigLIPVision(dim)
        self.language = nn.Embedding(vocab, dim)
        self.state = nn.Linear(action_dim, dim)
        self.action = nn.Linear(action_dim, dim)
        self.action_time = nn.Sequential(
            nn.Linear(2 * dim, dim),
            nn.SiLU(),
            nn.Linear(dim, dim),
        )
        self.layers = nn.ModuleList(
            [Pi0Layer(dim) for _ in range(18)]
        )
        self.suffix_final_norm = nn.RMSNorm(dim)
        self.out = nn.Linear(dim, action_dim)

    def forward(self, image, language, state, noisy_action, t):
        vision_tokens = self.vision(image)
        language_tokens = self.language(language) * math.sqrt(self.dim)
        prefix = torch.cat(
            [vision_tokens, language_tokens],
            dim=1,
        )

        state_token = self.state(state).unsqueeze(1)
        time = pi0_time_embedding(t, self.dim)
        time = time[:, None].expand(-1, self.horizon, -1)
        action_tokens = self.action(noisy_action)
        action_tokens = self.action_time(
            torch.cat([action_tokens, time], dim=-1)
        )
        suffix = torch.cat(
            [state_token, action_tokens],
            dim=1,
        )

        total_length = prefix.size(1) + suffix.size(1)
        valid = torch.ones(
            prefix.size(0),
            total_length,
            dtype=torch.bool,
            device=prefix.device,
        )
        prefix_ar = torch.zeros(
            prefix.size(0),
            prefix.size(1),
            dtype=torch.bool,
            device=prefix.device,
        )
        suffix_ar_row = torch.tensor(
            [True, True] + [False] * (self.horizon - 1),
            dtype=torch.bool,
            device=prefix.device,
        )
        suffix_ar = suffix_ar_row[None].expand(
            prefix.size(0),
            -1,
        )
        mask = block_attention_mask(
            valid,
            torch.cat([prefix_ar, suffix_ar], dim=1),
        )
        position = torch.cumsum(valid.long(), dim=1) - 1

        for layer in self.layers:
            prefix, suffix = layer(
                prefix,
                suffix,
                mask,
                position,
            )

        suffix = self.suffix_final_norm(suffix)
        return self.out(suffix[:, -self.horizon :])


@torch.no_grad()
def sample_pi0(model, image, language, state, steps=10):
    action = torch.randn(
        image.size(0),
        model.horizon,
        model.action_dim,
        device=image.device,
    )
    dt = -1.0 / steps

    for step in range(steps):
        t_value = 1.0 - step / steps
        t = torch.full(
            (image.size(0),),
            t_value,
            device=image.device,
        )
        velocity = model(
            image,
            language,
            state,
            action,
            t,
        )
        action = action + dt * velocity
    return action


pi0 = Pi0().to(device)
image = torch.randn(1, 3, 28, 28, device=device)
language = torch.randint(0, 64, (1, 3), device=device)
state = torch.randn(1, 3, device=device)
actions = torch.randn(1, 4, 3, device=device)
noise = torch.randn_like(actions)
t = torch.distributions.Beta(1.5, 1.0).sample((1,))
t = (t * 0.999 + 0.001).to(device)
x_t = t[:, None, None] * noise + (1 - t[:, None, None]) * actions
target_velocity = noise - actions
predicted_velocity = pi0(
    image,
    language,
    state,
    x_t,
    t,
)
F.mse_loss(predicted_velocity, target_velocity).backward()

sampled_actions = sample_pi0(
    pi0,
    image,
    language,
    state,
    steps=10,
)
assert len(pi0.vision.blocks) == 27
assert len(pi0.layers) == 18
assert pi0.layers[0].q_heads == 8
assert pi0.layers[0].kv_heads == 1
assert sampled_actions.shape == actions.shape


## 3. FAST preprocessing statistics are fitted on training data, not on each chunk


In [ ]:
def dct_matrix(length, device):
    time_index = torch.arange(
        length,
        device=device,
        dtype=torch.float32,
    )
    frequency_index = torch.arange(
        length,
        device=device,
        dtype=torch.float32,
    )[:, None]
    matrix = torch.cos(
        math.pi
        / length
        * (time_index + 0.5)
        * frequency_index
    )
    matrix[0] *= math.sqrt(1 / length)
    matrix[1:] *= math.sqrt(2 / length)
    return matrix


class QuantileActionNormalizer:
    def fit(self, action_corpus, low_q=0.01, high_q=0.99):
        flattened = action_corpus.reshape(
            -1,
            action_corpus.size(-1),
        )
        self.low = torch.quantile(
            flattened,
            low_q,
            dim=0,
        )
        self.high = torch.quantile(
            flattened,
            high_q,
            dim=0,
        )
        return self

    def transform(self, actions):
        return (
            2
            * (actions - self.low)
            / (self.high - self.low).clamp_min(1e-6)
            - 1
        ).clamp(-1, 1)

    def inverse(self, actions):
        return (
            0.5 * (actions + 1) * (self.high - self.low)
            + self.low
        )


## 4. Corpus-trained frozen BPE for DCT symbols

`fit()` sees the training corpus once and stores merge rules plus a fixed
vocabulary. `encode()` never changes those rules. Each DCT tensor has shape
`[frequency, action_dimension]`; ordinary row-major `reshape(-1)` therefore
interleaves action dimensions **inside each frequency**, matching FAST's
frequency-major flattening.


In [ ]:
def apply_merge_rule(sequence, pair, merged_symbol):
    output = []
    index = 0
    while index < len(sequence):
        matches = (
            index + 1 < len(sequence)
            and sequence[index] == pair[0]
            and sequence[index + 1] == pair[1]
        )
        if matches:
            output.append(merged_symbol)
            index += 2
        else:
            output.append(sequence[index])
            index += 1
    return output


def expand_symbol(symbol):
    if isinstance(symbol, tuple):
        left, right = symbol
        return expand_symbol(left) + expand_symbol(right)
    return [symbol]


class FrozenBPEActionTokenizer:
    def __init__(self, quantization_scale=64.0, merges=32):
        self.scale = quantization_scale
        self.max_merges = merges
        self.merge_rules = []
        self.symbol_to_id = None
        self.id_to_symbol = None
        self.min_token = None
        self.max_token = None

    def _quantized_frequency_symbols(self, actions):
        length = actions.size(1)
        matrix = dct_matrix(length, actions.device)
        coefficients = torch.einsum(
            "ft,btd->bfd",
            matrix,
            actions,
        )
        quantized = torch.round(
            coefficients * self.scale
        ).long()

        shifted = quantized - self.min_token
        if (shifted < 0).any():
            raise ValueError(
                "unseen coefficient is below training min_token"
            )
        alphabet_max = self.max_token - self.min_token
        if (shifted > alphabet_max).any():
            raise ValueError(
                "unseen coefficient is above training max_token"
            )

        # [batch, frequency, action_dim] -> frequency-major stream.
        return [
            row.reshape(-1).tolist()
            for row in shifted
        ]

    def fit(self, normalized_action_corpus):
        length = normalized_action_corpus.size(1)
        matrix = dct_matrix(
            length,
            normalized_action_corpus.device,
        )
        coefficients = torch.einsum(
            "ft,btd->bfd",
            matrix,
            normalized_action_corpus,
        )
        quantized = torch.round(
            coefficients * self.scale
        ).long()
        self.min_token = int(quantized.min().item())
        self.max_token = int(quantized.max().item())
        sequences = [
            row.reshape(-1).tolist()
            for row in quantized - self.min_token
        ]

        for merge_index in range(self.max_merges):
            pair_counts = Counter()
            for sequence in sequences:
                pair_counts.update(zip(sequence[:-1], sequence[1:]))
            if not pair_counts:
                break

            pair, count = pair_counts.most_common(1)[0]
            if count < 2:
                break
            merged_symbol = (pair[0], pair[1])
            self.merge_rules.append((pair, merged_symbol))
            sequences = [
                apply_merge_rule(
                    sequence,
                    pair,
                    merged_symbol,
                )
                for sequence in sequences
            ]

        symbols = set(
            range(self.max_token - self.min_token + 1)
        )
        for sequence in sequences:
            symbols.update(sequence)
        for pair, merged_symbol in self.merge_rules:
            symbols.update(pair)
            symbols.add(merged_symbol)

        ordered = sorted(symbols, key=repr)
        self.symbol_to_id = {
            symbol: index
            for index, symbol in enumerate(ordered)
        }
        self.id_to_symbol = {
            index: symbol
            for symbol, index in self.symbol_to_id.items()
        }
        return self

    def encode(self, normalized_actions):
        if self.symbol_to_id is None:
            raise RuntimeError("fit() must be called before encode()")

        encoded_batch = []
        for sequence in self._quantized_frequency_symbols(
            normalized_actions
        ):
            for pair, merged_symbol in self.merge_rules:
                sequence = apply_merge_rule(
                    sequence,
                    pair,
                    merged_symbol,
                )
            encoded_batch.append(
                [self.symbol_to_id[symbol] for symbol in sequence]
            )
        return encoded_batch

    def decode_quantized(self, token_ids, shape, device):
        batch, frequencies, action_dim = shape
        rows = []
        for encoded in token_ids:
            base_symbols = []
            for token_id in encoded:
                symbol = self.id_to_symbol[token_id]
                base_symbols.extend(expand_symbol(symbol))
            row = torch.tensor(
                base_symbols,
                dtype=torch.long,
                device=device,
            )
            rows.append(
                row.view(frequencies, action_dim)
            )
        shifted = torch.stack(rows)
        return shifted + self.min_token

    def decode_actions(self, token_ids, shape, device):
        quantized = self.decode_quantized(
            token_ids,
            shape,
            device,
        )
        coefficients = quantized.float() / self.scale
        frequencies = shape[1]
        matrix = dct_matrix(frequencies, device)
        return torch.einsum(
            "tf,bfd->btd",
            matrix.T,
            coefficients,
        )


## 5. Fit once, encode unseen chunks, decode with frozen rules


In [ ]:
training_actions = torch.randn(32, 8, 3, device=device)
held_out_actions = torch.randn(2, 8, 3, device=device)

normalizer = QuantileActionNormalizer().fit(training_actions)
normalized_training = normalizer.transform(training_actions)
normalized_held_out = normalizer.transform(held_out_actions)

fast_tokenizer = FrozenBPEActionTokenizer(
    quantization_scale=64.0,
    merges=32,
).fit(normalized_training)

rules_before = list(fast_tokenizer.merge_rules)
vocabulary_before = dict(fast_tokenizer.symbol_to_id)
encoded = fast_tokenizer.encode(normalized_held_out)

# Encoding new data must not train or mutate the tokenizer.
assert fast_tokenizer.merge_rules == rules_before
assert fast_tokenizer.symbol_to_id == vocabulary_before

shape = (
    normalized_held_out.size(0),
    normalized_held_out.size(1),
    normalized_held_out.size(2),
)
reconstructed_normalized = fast_tokenizer.decode_actions(
    encoded,
    shape,
    normalized_held_out.device,
)
reconstructed_actions = normalizer.inverse(
    reconstructed_normalized
)

assert reconstructed_actions.shape == held_out_actions.shape
assert torch.isfinite(reconstructed_actions).all()
assert len(fast_tokenizer.merge_rules) > 0

# Explicit frequency-major ordering check on a known [F, D] tensor.
known = torch.tensor(
    [[[0, 1], [10, 11], [20, 21]]],
    dtype=torch.long,
)
assert known.reshape(-1).tolist() == [0, 1, 10, 11, 20, 21]

print("pi0 sampled actions:", sampled_actions.shape)
print("FAST frozen merges:", len(fast_tokenizer.merge_rules))
print("FAST held-out token lengths:", [len(x) for x in encoded])


## Structural checklist

FAST no longer trains a tiny merger independently for every action chunk and no
longer transposes `[frequency, action_dimension]` before flattening. Training
statistics and BPE merge rules are fitted once on a corpus, then frozen for
held-out encoding/decoding. π0 retains its block-mask and flow-matching path.
